# 🚀 YOLOv5 Face Mask Detection on Colab
**在 Colab GPU 上训练口罩检测模型**

1. 下载 Kaggle Face Mask Detection 数据集
2. 转换为 YOLO 格式
3. 训练 YOLOv5
4. 运行推理检测

## 1. 初始化环境

In [ ]:
# 挂载 Google Drive（保存训练结果用）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 检查 GPU
!nvidia-smi

In [ ]:
# 克隆你自己的 YOLOv5 仓库
!git clone https://github.com/qianz7884-blip/yolo.git
%cd yolo
!pip install -r requirements.txt -q

## 2. 下载并准备数据集

In [ ]:
# 安装 kagglehub 下载数据集
!pip install kagglehub -q

In [ ]:
import xml.etree.ElementTree as ET
import os
import shutil
import random
from pathlib import Path

import kagglehub

# 下载数据集
print("正在下载 Face Mask Detection 数据集...")
src = Path(kagglehub.dataset_download('andrewmvd/face-mask-detection'))
print(f"下载位置: {src}")

# 创建 YOLOv5 格式目录
dst = Path('/content/datasets/mask')
for split in ['train', 'val']:
    (dst / 'images' / split).mkdir(parents=True, exist_ok=True)
    (dst / 'labels' / split).mkdir(parents=True, exist_ok=True)

# 类别映射 (必须与 mask.yaml 一致)
CLASS_MAP = {'with_mask': 0, 'without_mask': 1, 'mask_weared_incorrect': 2}

# 解析所有 VOC XML 标注
all_data = []
for xml_path in sorted((src / 'annotations').glob('*.xml')):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    filename = root.find('filename').text
    size = root.find('size')
    w, h = int(size.find('width').text), int(size.find('height').text)
    
    objects = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in CLASS_MAP:
            continue
        cls_id = CLASS_MAP[name]
        bbox = obj.find('bndbox')
        xmin, ymin = int(bbox.find('xmin').text), int(bbox.find('ymin').text)
        xmax, ymax = int(bbox.find('xmax').text), int(bbox.find('ymax').text)
        # VOC → YOLO (归一化)
        xc = ((xmin + xmax) / 2) / w
        yc = ((ymin + ymax) / 2) / h
        bw = (xmax - xmin) / w
        bh = (ymax - ymin) / h
        objects.append(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    
    all_data.append({'filename': filename, 'objects': objects})

# 划分训练/验证集 (80/20)
random.seed(42)
random.shuffle(all_data)
split_idx = int(len(all_data) * 0.8)

for split_name, data in [('train', all_data[:split_idx]), ('val', all_data[split_idx:])]:
    for item in data:
        stem = Path(item['filename']).stem
        # 复制图片
        src_img = src / 'images' / item['filename']
        if src_img.exists():
            shutil.copy2(src_img, dst / 'images' / split_name / item['filename'])
        # 写入 YOLO 标签
        lbl = dst / 'labels' / split_name / f'{stem}.txt'
        with open(lbl, 'w') as f:
            if item['objects']:
                f.write('\n'.join(item['objects']) + '\n')

print(f"完成! 训练集: {len(all_data[:split_idx])}张, 验证集: {len(all_data[split_idx:])}张")

# 统计类别
counts = {0: 0, 1: 0, 2: 0}
for d in all_data:
    for obj in d['objects']:
        counts[int(obj[0])] += 1
for k, v in CLASS_MAP.items():
    print(f"  {v} ({k}): {counts[v]}")

## 3. 创建数据集配置文件

In [ ]:
# 写入 mask.yaml
yaml_content = """
path: /content/datasets/mask
train: images/train/
val: images/val/

names:
  0: with_mask
  1: without_mask
  2: mask_weared_incorrect
"""

with open('/content/yolov5/data/mask.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("✅ mask.yaml 已创建")

## 4. 开始训练

In [ ]:
# 训练 YOLOv5s（轻量快速），想要更高精度可换成 yolov5m / yolov5l
!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 100 \
    --data mask.yaml \
    --weights yolov5s.pt \
    --project /content/drive/MyDrive/yolov5-mask \
    --name exp \
    --workers 2

## 5. 查看训练结果

In [ ]:
# 显示训练结果
from IPython.display import Image, display

exp_dir = '/content/drive/MyDrive/yolov5-mask/exp'
print("📊 混淆矩阵:")
display(Image(filename=f'{exp_dir}/confusion_matrix.png'))
print("📈 训练曲线:")
display(Image(filename=f'{exp_dir}/results.png'))

In [ ]:
# 查看验证集批量预测结果
print("🔍 验证集检测样张:")
display(Image(filename=f'{exp_dir}/val_batch0_pred.jpg'))
display(Image(filename=f'{exp_dir}/val_batch1_pred.jpg'))

## 6. 用训练好的模型做推理

In [ ]:
# 上传你自己的图片或用测试集图片做检测
from google.colab import files
import glob

# 方式一: 上传图片
uploaded = files.upload()

for fname in uploaded.keys():
    !python detect.py \
        --weights /content/drive/MyDrive/yolov5-mask/exp/weights/best.pt \
        --source {fname} \
        --conf 0.25 \
        --save-txt
    
    # 显示结果
    result = glob.glob('/content/yolov5/runs/detect/*/' + fname)
    if result:
        display(Image(filename=result[0]))

In [ ]:
# 方式二: 对验证集全部图片做批量检测
!python detect.py \
    --weights /content/drive/MyDrive/yolov5-mask/exp/weights/best.pt \
    --source /content/datasets/mask/images/val/ \
    --conf 0.25 \
    --save-txt \
    --project /content/drive/MyDrive/yolov5-mask \
    --name detect_val \
    --nosave  # 如果只想看统计信息，不想保存所有图片

## 7. 下载训练好的模型

In [ ]:
# 下载 best.pt 到本地
from google.colab import files
files.download('/content/drive/MyDrive/yolov5-mask/exp/weights/best.pt')
files.download('/content/drive/MyDrive/yolov5-mask/exp/results.png')

---
### 📋 参数说明

| 参数 | 含义 | 建议 |
|------|------|------|
| `--img 640` | 输入图片尺寸 | 显卡好可以 960/1280 |
| `--batch 16` | 批大小 | T4 用 16, V100/A100 用 32-64 |
| `--epochs 100` | 训练轮数 | 小数据集建议 100-300 |
| `--weights yolov5s.pt` | 预训练权重 | s/m/l/x 越大越准越慢 |
| `--conf 0.25` | 检测置信度阈值 | 越小检出越多(误检也多) |

### 💡 提示
- Colab 免费 GPU 约可用 3-12 小时
- 模型自动保存到 Google Drive 不会丢失
- `best.pt` = 验证集上 mAP 最高的模型
- `last.pt` = 最后一轮训练的模型